# 読みから表記を引く辞書をコーパスから作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_japanese_input/kkc_dictionary.ipynb)

NLP クックブック 27-03 ／ 日本語入力 (かな漢字変換)

かな漢字変換のための「読み → 表記の候補」の辞書を作りたい。入力の先頭から、辞書にある読みをすべて高速に見つけたい。

In [1]:
!pip install -q fugashi unidic-lite jaconv marisa-trie

In [2]:
import io, re, urllib.request, zipfile
from collections import Counter, defaultdict
import jaconv
from fugashi import Tagger

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

# コーパスを形態素解析して、(表記, 読み) の列にする。読みは形態素解析器が付けたものを使う
tagger = Tagger()
base = "https://www.aozora.gr.jp/cards/000148/files/"
sentences = []
for f in ["789_ruby_5639.zip", "752_ruby_2438.zip", "773_ruby_5968.zip"]:      # 吾輩は猫である・坊っちゃん・こころ
    for s in re.split(r"(?<=。)|\n", aozora(base + f)):
        if s.strip():
            sentences.append([(w.surface, jaconv.kata2hira(w.feature.kana or w.surface)) for w in tagger(s.strip())])
print(len(sentences), "文", sum(map(len, sentences)), "語")
print(sentences[2])

16959 文 375235 語
[('名前', 'なまえ'), ('は', 'は'), ('まだ', 'まだ'), ('無い', 'ない'), ('。', '。')]


In [3]:
# 辞書: 読み → {表記: 出現回数}
dic = defaultdict(Counter)
for sent in sentences:
    for surface, reading in sent:
        dic[reading][surface] += 1
print(len(dic), "種類の読み")
for reading in ["こうしょう", "かんじ", "きく", "は", "かける"]:
    print(reading, dic[reading].most_common(6))

ambiguity = Counter(len(v) for v in dic.values())
print("表記の候補が 1 つだけの読み:", ambiguity[1], "/ 2 つ以上:", sum(c for n, c in ambiguity.items() if n > 1))

13540 種類の読み
こうしょう [('交渉', 6), ('高尚', 6), ('高声', 2), ('工廠', 2)]
かんじ [('感じ', 96), ('幹事', 3), ('観じ', 2)]
きく [('聞く', 155), ('きく', 14), ('利く', 8), ('聴く', 7), ('菊', 3)]
は [('は', 12409), ('ハ', 36), ('歯', 19), ('葉', 13), ('派', 3), ('刃', 3)]
かける [('かける', 25), ('掛ける', 8), ('懸ける', 5), ('欠ける', 1), ('掻ける', 1), ('書ける', 1)]
表記の候補が 1 つだけの読み: 10890 / 2 つ以上: 2650


In [4]:
# トライ木に入れて、入力の各位置から始まる辞書の読みをすべて列挙する (共通接頭辞検索)
import marisa_trie

trie = marisa_trie.Trie(dic.keys())
text = "わがはいはねこである"
for i in range(len(text)):
    print(f"{i:2} {text[i:]:　<10} {trie.prefixes(text[i:])}")

 0 わがはいはねこである ['わ', 'わが', 'わがはい']
 1 がはいはねこである　 ['が']
 2 はいはねこである　　 ['は', 'はい']
 3 いはねこである　　　 ['い']
 4 はねこである　　　　 ['は', 'はね']
 5 ねこである　　　　　 ['ね', 'ねこ']
 6 こである　　　　　　 ['こ']
 7 である　　　　　　　 ['で']
 8 ある　　　　　　　　 ['あ', 'ある']
 9 る　　　　　　　　　 ['る']


In [5]:
import time

t = time.perf_counter()
n = sum(len(trie.prefixes(text[i:])) for _ in range(10000) for i in range(len(text)))
print(f"共通接頭辞検索 {10000 * len(text)} 回: {time.perf_counter() - t:.2f} 秒")
trie.save("reading.trie")

共通接頭辞検索 100000 回: 0.07 秒


- かな漢字変換の辞書は「読みをキーにして表記の候補を引く」もの。形態素解析の辞書 (表記をキーにして読みや品詞を引く) の逆向きにあたる。
- 同じ読みに多くの表記がある (同音異義語) ことが、かな漢字変換の難しさの中心。どれを選ぶかは、頻度と前後の文脈で決める。
- 共通接頭辞検索は「この位置から始まる単語の候補をすべて挙げる」操作で、次項のラティス (単語の候補のグラフ) を作るのに使う。トライ木なら入力の長さに比例した時間で済む。
- 実用の辞書は数十万〜数百万語。Mozc の辞書、SKK 辞書、Sudachi や UniDic の語彙に読みを付けたものなどが元になる。